# RAG 2: przykład średnio złożony

Tutaj przechodzimy poziom wyżej:
- mamy dłuższe dokumenty,
- dzielimy je na **chunki**,
- wyszukujemy najlepsze chunki,
- budujemy odpowiedź z cytowanymi źródłami.

To jest bliżej realnych systemów RAG.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import re

In [2]:
long_documents = [
    {
        "doc_id": "nlp_lecture",
        "title": "Notatki z wykładu NLP",
        "text": (
            "Przetwarzanie języka naturalnego obejmuje tokenizację, normalizację, reprezentację tekstu oraz modelowanie semantyczne. "
            "W klasycznych systemach wykorzystywano worki słów, TF-IDF i modele statystyczne. "
            "Współczesne NLP jest silnie związane z embeddingami i transformerami. "
            "Modele językowe potrafią rozumieć kontekst zdania znacznie lepiej niż wcześniejsze architektury. "
            "RAG wykorzystuje zewnętrzną bazę wiedzy, aby ograniczyć halucynacje modelu i zwiększyć trafność odpowiedzi."
        )
    },
    {
        "doc_id": "ml_notes",
        "title": "Notatki z uczenia maszynowego",
        "text": (
            "Uczenie maszynowe można podzielić na uczenie nadzorowane, nienadzorowane i ze wzmocnieniem. "
            "Klasyfikacja i regresja należą do najczęściej stosowanych zadań nadzorowanych. "
            "W pracy z tekstem często wykorzystuje się klasyfikację sentymentu, analizę tematyczną i wyszukiwanie semantyczne. "
            "Modele wektorowe pozwalają liczyć podobieństwo między dokumentami i zapytaniami."
        )
    },
    {
        "doc_id": "rag_notes",
        "title": "Notatki o systemach RAG",
        "text": (
            "Pipeline RAG zwykle składa się z indeksowania dokumentów, podziału na fragmenty, obliczenia reprezentacji i etapu retrieval. "
            "Następnie wybrane fragmenty trafiają do generatora odpowiedzi. "
            "Jakość chunkingu ma ogromne znaczenie, bo zbyt krótkie fragmenty tracą sens, a zbyt długie rozmywają informację. "
            "Dodatkowo można stosować reranking, filtrowanie metadanych oraz ocenę jakości odpowiedzi."
        )
    }
]

In [3]:
def chunk_text(text, chunk_size=180, overlap=40):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

rows = []
for doc in long_documents:
    for i, chunk in enumerate(chunk_text(doc["text"], chunk_size=180, overlap=40)):
        rows.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{i}",
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "chunk_text": chunk
        })

chunks_df = pd.DataFrame(rows)
chunks_df.head(10)

,chunk_id,doc_id,title,chunk_text
0,nlp_lecture_chunk_0,nlp_lecture,Notatki z wykładu NLP,Przetwarzanie języka naturalnego obejmuje toke...
1,nlp_lecture_chunk_1,nlp_lecture,Notatki z wykładu NLP,"ach wykorzystywano worki słów, TF-IDF i modele..."
2,nlp_lecture_chunk_2,nlp_lecture,Notatki z wykładu NLP,zykowe potrafią rozumieć kontekst zdania znacz...
3,nlp_lecture_chunk_3,nlp_lecture,Notatki z wykładu NLP,zyć halucynacje modelu i zwiększyć trafność od...
4,ml_notes_chunk_0,ml_notes,Notatki z uczenia maszynowego,Uczenie maszynowe można podzielić na uczenie n...
5,ml_notes_chunk_1,ml_notes,Notatki z uczenia maszynowego,osowanych zadań nadzorowanych. W pracy z tekst...
6,ml_notes_chunk_2,ml_notes,Notatki z uczenia maszynowego,zne. Modele wektorowe pozwalają liczyć podobie...
7,rag_notes_chunk_0,rag_notes,Notatki o systemach RAG,Pipeline RAG zwykle składa się z indeksowania ...
8,rag_notes_chunk_1,rag_notes,Notatki o systemach RAG,ne fragmenty trafiają do generatora odpowiedzi...
9,rag_notes_chunk_2,rag_notes,Notatki o systemach RAG,ozmywają informację. Dodatkowo można stosować ...


In [4]:
vectorizer = TfidfVectorizer()
chunk_matrix = vectorizer.fit_transform(chunks_df["chunk_text"])

def retrieve_chunks(query, top_k=3):
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, chunk_matrix).flatten()
    idxs = sims.argsort()[::-1][:top_k]
    out = []
    for idx in idxs:
        row = chunks_df.iloc[idx]
        out.append({
            "chunk_id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "title": row["title"],
            "chunk_text": row["chunk_text"],
            "score": float(sims[idx])
        })
    return out

In [5]:
def answer_with_sources(query, retrieved_chunks):
    bullet_points = []
    for item in retrieved_chunks:
        bullet_points.append(
            f"- [{item['doc_id']}] {item['chunk_text']}"
        )
    context = "\n".join(bullet_points)
    answer = (
        f"Pytanie: {query}\n\n"
        "Odpowiedź robocza:\n"
        "Na podstawie odnalezionych fragmentów można stwierdzić, że RAG łączy wyszukiwanie kontekstu "
        "z generowaniem odpowiedzi, a jakość odpowiedzi zależy m.in. od chunkingu, reprezentacji tekstu "
        "oraz doboru właściwych fragmentów.\n\n"
        f"Kontekst:\n{context}"
    )
    return answer

In [6]:
query = "Dlaczego chunking jest ważny w systemach RAG?"
retrieved = retrieve_chunks(query, top_k=3)

pd.DataFrame(retrieved)

,chunk_id,doc_id,title,chunk_text,score
0,nlp_lecture_chunk_0,nlp_lecture,Notatki z wykładu NLP,Przetwarzanie języka naturalnego obejmuje toke...,0.151234
1,nlp_lecture_chunk_1,nlp_lecture,Notatki z wykładu NLP,"ach wykorzystywano worki słów, TF-IDF i modele...",0.140085
2,nlp_lecture_chunk_2,nlp_lecture,Notatki z wykładu NLP,zykowe potrafią rozumieć kontekst zdania znacz...,0.101813


In [7]:
print(answer_with_sources(query, retrieved))

Pytanie: Dlaczego chunking jest ważny w systemach RAG?

Odpowiedź robocza:
Na podstawie odnalezionych fragmentów można stwierdzić, że RAG łączy wyszukiwanie kontekstu z generowaniem odpowiedzi, a jakość odpowiedzi zależy m.in. od chunkingu, reprezentacji tekstu oraz doboru właściwych fragmentów.

Kontekst:
- [nlp_lecture] Przetwarzanie języka naturalnego obejmuje tokenizację, normalizację, reprezentację tekstu oraz modelowanie semantyczne. W klasycznych systemach wykorzystywano worki słów, TF-IDF i 
- [nlp_lecture] ach wykorzystywano worki słów, TF-IDF i modele statystyczne. Współczesne NLP jest silnie związane z embeddingami i transformerami. Modele językowe potrafią rozumieć kontekst zdania
- [nlp_lecture] zykowe potrafią rozumieć kontekst zdania znacznie lepiej niż wcześniejsze architektury. RAG wykorzystuje zewnętrzną bazę wiedzy, aby ograniczyć halucynacje modelu i zwiększyć trafn


## Co tu jest ważne dydaktycznie
W tym notebooku pojawiają się elementy, które studenci spotkają w praktyce:
- podział dokumentu na fragmenty,
- indeksowanie chunków zamiast całych dokumentów,
- źródła w odpowiedzi,
- wpływ jakości chunkingu na jakość RAG.